# Week 3 - Text Preprocessing (2/2)
**Notebook 2 : Visualization**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

After preprocessing, the first thing we usually do is **EDA (Exploratory Data Analysis)** on the text:
which words appear most often, and what does the vocabulary look like?

In this notebook we apply the full preprocessing pipeline from notebook 1 to a real book (*Moby Dick*) and visualise the result with

1. a **word-frequency graph** (line / bar chart), and
2. a **word cloud** (plain, frequency-based and image-masked).

## &nbsp;0. Setup

In [ ]:
import nltk

for r in ['gutenberg', 'punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords',
          'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(r, quiet=True)

# wordcloud is NOT pre-installed in Colab
!pip install -q wordcloud

In [ ]:
# Sharper figures in the notebook (retina display setting)

%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt

%matplotlib inline

## &nbsp;1. Gutenberg Corpus
NLTK ships a small sample of the **Project Gutenberg** collection (free classic e-books). Let's list the available titles and load *Moby Dick* by Herman Melville.

In [ ]:
from nltk.corpus import gutenberg

file_names = gutenberg.fileids()    # titles available in the corpus
print(file_names)

In [ ]:
doc_mobydick = gutenberg.open('melville-moby_dick.txt').read()
doc_mobydick = doc_mobydick.lower()          # lowercase so that 'Whale' and 'whale' are counted as the same word

print('# Num of characters used:', len(doc_mobydick))
print('# Text sample:')
print(doc_mobydick[:500])

## &nbsp;2. Preprocessing the Book
We repeat the steps from notebook 1 and observe **how the number of tokens changes** at each step.
Stemming and lemmatization are shown for comparison only; the pipeline we actually use
continues with `RegexpTokenizer` and stop-word removal.

In [ ]:
# word tokenization

from nltk.tokenize import word_tokenize

tokens_mobydick = word_tokenize(doc_mobydick)

print('# Num of tokens used:', len(tokens_mobydick))
print('# Token sample:', tokens_mobydick[:20])

In [ ]:
# stemming (number of tokens stays the same)

from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

stem_tokens_mobydick = [stemmer.stem(token) for token in tokens_mobydick]   

print('# Num of tokens after stemming:', len(stem_tokens_mobydick))
print('# Token sample:', stem_tokens_mobydick[:20])

In [ ]:
# lemmatization

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
lem_tokens_mobydick = [lemmatizer.lemmatize(token) for token in tokens_mobydick]   

print('# Num of tokens after lemmatization:', len(lem_tokens_mobydick))
print('# Token sample:', lem_tokens_mobydick[:20])

In [ ]:
# words of 3+ characters → punctuation and very short words disappear

from nltk.tokenize import RegexpTokenizer

tokenizer = RegexpTokenizer(r"[\w']{3,}")           
reg_tokens_mobydick = tokenizer.tokenize(doc_mobydick)

print('# Num of tokens with RegexpTokenizer:', len(reg_tokens_mobydick))
print('# Token sample:', reg_tokens_mobydick[:20])

In [ ]:
# stop-word removal

from nltk.corpus import stopwords

english_stops = set(stopwords.words('english'))
result_mobydick = [word for word in reg_tokens_mobydick if word not in english_stops]   

print('# Num of tokens after stopword elimination:', len(result_mobydick))
print('# Token sample:', result_mobydick[:20])

## &nbsp;3. Word Frequency

### &nbsp;3-1. Counting with a dictionary
We count how many times each word occurs, then sort the vocabulary by frequency.

In [ ]:
mobydick_word_count = dict()

for word in result_mobydick:
    mobydick_word_count[word] = mobydick_word_count.get(word, 0) + 1                        # +1 for every occurrence
print('# Num of used words (vocabulary size):', len(mobydick_word_count))

sorted_word_count = sorted(mobydick_word_count, key=mobydick_word_count.get, reverse=True)  # words sorted by frequency
print("# Top 20 high frequency words:")

for key in sorted_word_count[:20]:
    print(f'{repr(key)}: {mobydick_word_count[key]}', end=', ')

### &nbsp;3-2. FreqDist
`FreqDist` does the same job in one line and offers `.most_common()` and `.plot()`.

In [ ]:
from nltk import FreqDist

freq = FreqDist(result_mobydick)
print(freq.most_common(20))

# Look at the list : `one`, `like`, `upon`, `would` are frequent but tell us nothing about the book.
# Stop-word removal did not catch them. We can tag the POS and keep only nouns, verbs and adjectives.

### &nbsp;3-3. POS Filtering
Tag the POS and keep only nouns, verbs and adjectives (`NN`, `VB`, `VBD`, `JJ`).

In [ ]:
my_tag_set = ['NN', 'VB', 'VBD', 'JJ']
my_words = [word for word, tag in nltk.pos_tag(result_mobydick) if tag in my_tag_set]

mobydick_word_count = dict()

for word in my_words:
    mobydick_word_count[word] = mobydick_word_count.get(word, 0) + 1
print('# Num of used words:', len(mobydick_word_count))

sorted_word_count = sorted(mobydick_word_count, key=mobydick_word_count.get, reverse=True)
print("# Top 20 high frequency words:")
for key in sorted_word_count[:20]:
    print(f'{repr(key)}: {mobydick_word_count[key]}', end=', ')

## &nbsp;4. Frequency Graphs

### &nbsp;4-1. Zipf Curve
Plotting the frequency of every word in rank order shows the typical **long-tail** shape: a few words are extremely frequent and most words are rare.

In [ ]:
# frequencies in rank order

w = [mobydick_word_count[key] for key in sorted_word_count]   

plt.figure(figsize=(8, 4))
plt.plot(w)
plt.xlabel('rank'); plt.ylabel('frequency'); plt.title('Word frequency by rank (Moby Dick)')
plt.show()

In [ ]:
# Log-log scale makes the long tail easier to see

plt.figure(figsize=(8, 4))
plt.loglog(w)
plt.xlabel('rank (log)'); plt.ylabel('frequency (log)'); plt.title('Zipf plot')
plt.show()

### &nbsp;4-2. Bar Charts

In [ ]:
n = sorted_word_count[:20]                       # top 20 words
w = [mobydick_word_count[key] for key in n]      # their frequencies

plt.figure(figsize=(12, 4))
plt.bar(range(len(n)), w, tick_label=n)          # vertical bar chart
plt.xticks(rotation=45)
plt.title('Top 20 words (after POS filtering)')
plt.show()

In [ ]:
n = sorted_word_count[:20][::-1]                 # reverse so that the most frequent word is on top
w = [mobydick_word_count[key] for key in n]

plt.figure(figsize=(8, 6))
plt.barh(range(len(n)), w, tick_label=n)         # horizontal bar chart
plt.title('Top 20 words (after POS filtering)')
plt.show()

In [ ]:
# FreqDist has a built-in plot as well
# Note: 'freq' was built from result_mobydick - BEFORE the POS filter, unlike the two charts above.

plt.figure(figsize=(12, 4))
freq.plot(30, cumulative=False, title='Word Frequency – Moby Dick (top 30, before POS filtering)')
plt.show()

## &nbsp;5. Word Cloud
A **word cloud** draws frequent words larger. It is a quick way to show the key themes of a text to non-experts.

Main options of `WordCloud(...)`:

| option | meaning |
|---|---|
| `max_words` | maximum number of words shown |
| `max_font_size` | size of the most frequent word |
| `background_color` | e.g. `'white'` |
| `stopwords` | a set of words to ignore |
| `collocations` | `True` (default) also shows frequent **bigrams** such as `sperm whale` |
| `mask` | a NumPy image array; the cloud is drawn inside the non-white area |

### &nbsp;5-1. From Raw Text
`generate(text)` does its own simple tokenization and stop-word removal.

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud().generate(doc_mobydick)   # generate a word cloud image directly from the text

plt.figure(figsize=(8, 4))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

In [ ]:
wordcloud.to_array().shape   # the cloud is just an RGB image (height, width, 3)

### &nbsp;5-2. From Frequencies
`generate_from_frequencies(dict)` uses the counts we computed after preprocessing (POS-filtered), so the picture reflects **our** pipeline.

In [ ]:
wordcloud = WordCloud(max_font_size=60, background_color='white').generate_from_frequencies(mobydick_word_count)

plt.figure(figsize=(8, 4))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

### &nbsp;5-3. Image Mask
We draw the cloud inside a whale silhouette. The mask image is downloaded from the course GitHub repository
(you can also use any other black-and-white silhouette image).

In [ ]:
# Download the mask image (a PNG with a white background) from the course repository
!wget -q -O whale_mask.png "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week03-text-preprocessing/data/whale_mask.png"

import os
print("downloaded:", os.path.getsize("whale_mask.png"), "bytes")    # should be > 0

# If the download fails, you can upload the file by hand:
# open the Files panel on the left (folder icon) and drag whale_mask.png into it.

In [ ]:
import numpy as np
from PIL import Image

mobydick_mask = np.array(Image.open("whale_mask.png"))   # load the image as a NumPy array
print(mobydick_mask.shape)

wc = WordCloud(background_color="white",                 # background color
               max_words=30,                             # show only the 30 most frequent words
               mask=mobydick_mask,                       # draw inside the mask
               contour_width=3,                          # outline width
               contour_color='steelblue')                # outline color

wc.generate_from_frequencies(mobydick_word_count)
wc.to_file("mobydick_wordcloud.png")                     # save the image (check the Files panel on the left in Colab)

plt.figure(figsize=(10, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis("off")
plt.show()

### &nbsp;5-4. Bigram Cloud
`WordCloud` mixes in some word pairs automatically (`collocations=True` by default).
To control this yourself, build the bigrams explicitly and pass their frequencies.

In [ ]:
from nltk import bigrams
from collections import Counter

# Note: bigrams are built from result_mobydick - BEFORE the POS filter.
bigram_list = ["_".join(bg) for bg in bigrams(result_mobydick)]    # e.g. 'sperm_whale'
bigram_freq = Counter(bigram_list)
print(bigram_freq.most_common(10))

bigram_wc = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(bigram_freq)

plt.figure(figsize=(10, 5))
plt.imshow(bigram_wc, interpolation='bilinear')
plt.axis("off")
plt.show()

### ✅ What we practiced
* Corpus loading : NLTK `gutenberg`, watching the token count shrink through the pipeline
* Word counting : dictionary, `FreqDist`, POS filtering
* Frequency graphs : rank curve (Zipf), bar / barh charts, `FreqDist.plot`
* Word clouds : from text, from frequencies, with an image mask, bigram clouds

Next: **Assignment 1** applies exactly these steps to a Twitter sentiment dataset.